In [ ]:
import warnings
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

In [ ]:
import os
DATA_PATH = os.environ.get(
    "MONETARY_DATA_PATH", os.path.join("..", "data", "Monetary_transmission_data_III.xlsx")
)
FEATURE_COLS = ["CPI", "EXR", "M2b", "RSV", "BRNT"]
TARGET_COLS = ["CPI", "EXR"]
TEST_FRAC = 0.15
MIN_TEST = 4
P_RANGE = range(0, 3)
D_RANGE = range(0, 2)
Q_RANGE = range(0, 3)

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load raw Excel data and enforce a sorted monthly DatetimeIndex."""
    df = pd.read_excel(path, parse_dates=["DATE"])
    return df.sort_values("DATE").set_index("DATE").asfreq("MS")


df = load_data(DATA_PATH)[FEATURE_COLS]
df.head()

In [ ]:
# Same structural-break partition as the VAR/VECM and neural-net notebooks, so all
# five models are compared on identical windows. "W5" is the full, unsegmented series.
BREAK_DATES = pd.to_datetime(["2008-06-01", "2016-10-01", "2022-03-01"])
EDGES = [df.index.min(), *BREAK_DATES, df.index.max() + pd.DateOffset(days=1)]

WINDOWS = {f"W{i + 1}": (EDGES[i], EDGES[i + 1] - pd.DateOffset(days=1)) for i in range(len(EDGES) - 1)}
WINDOWS["W5"] = (df.index.min(), df.index.max())

for label, (start, end) in WINDOWS.items():
    print(f"{label}: {start.date()} -> {end.date()} ({len(df.loc[start:end])} months)")

In [ ]:
def select_order(series: pd.Series) -> tuple:
    """Grid-search ARIMA(p, d, q) by AIC on a single series."""
    best_aic, best_order = np.inf, (1, 1, 0)
    for order in product(P_RANGE, D_RANGE, Q_RANGE):
        try:
            aic = ARIMA(series, order=order).fit().aic
        except Exception:
            continue
        if aic < best_aic:
            best_aic, best_order = aic, order
    return best_order


def fit_forecast_series(series: pd.Series, test_size: int) -> tuple:
    """Select the AIC-optimal order on train, forecast the test horizon."""
    train, test = series.iloc[:-test_size], series.iloc[-test_size:]
    order = select_order(train)
    fitted = ARIMA(train, order=order).fit()
    forecast = fitted.forecast(steps=test_size)
    return forecast, test, order

In [ ]:
def run_window_pipeline(df_window: pd.DataFrame) -> dict:
    """Fit and evaluate one ARIMA model per target series within a structural-break window."""
    test_size = max(int(len(df_window) * TEST_FRAC), MIN_TEST)
    window_result = {}
    for col in TARGET_COLS:
        forecast, actual, order = fit_forecast_series(df_window[col], test_size)
        window_result[col] = {
            "order": order,
            "forecast": forecast,
            "actual": actual,
            "RMSE": np.sqrt(mean_squared_error(actual, forecast)),
            "MAE": mean_absolute_error(actual, forecast),
        }
    return window_result


results = {}
for label, (start, end) in WINDOWS.items():
    df_w = df.loc[start:end]
    results[label] = run_window_pipeline(df_w)
    print(f"{label}: n={len(df_w)}")
    for col in TARGET_COLS:
        r = results[label][col]
        print(f"  {col}: order={r['order']} RMSE={r['RMSE']:.4f} MAE={r['MAE']:.4f}")

In [ ]:
# Selected ARIMA order and held-out test error per window/target.
summary_df = pd.DataFrame({
    (label, col): {"order": r[col]["order"], "RMSE": r[col]["RMSE"], "MAE": r[col]["MAE"]}
    for label, r in results.items() for col in TARGET_COLS
}).T
summary_df.index.names = ["window", "target"]
summary_df

In [ ]:
fig, axes = plt.subplots(len(TARGET_COLS), len(WINDOWS), figsize=(5 * len(WINDOWS), 4 * len(TARGET_COLS)), sharey="row")
for col_idx, target in enumerate(TARGET_COLS):
    for win_idx, (label, r) in enumerate(results.items()):
        ax = axes[col_idx, win_idx]
        actual, forecast = r[target]["actual"], r[target]["forecast"]
        ax.plot(actual.index, actual, label="Actual")
        ax.plot(actual.index, forecast, label="Predicted")
        ax.set_title(f"{target} | {label}\nARIMA{r[target]['order']}", fontsize=9)
        ax.tick_params(axis="x", labelrotation=45)
axes[0, 0].legend()
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("../results", exist_ok=True)
# Standardized long-format export consumed by the model-comparison pipeline.
export_rows = [
    {"window": label, "target": col, "RMSE": r[col]["RMSE"], "MAE": r[col]["MAE"]}
    for label, r in results.items() for col in TARGET_COLS
]
pd.DataFrame(export_rows).to_csv(f"../results/metrics_arima.csv", index=False)